In [218]:
%pip install pandas nltk sacrebleu rouge-score sentence-transformers bert-score numpy matplotlib

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
88847.01s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [219]:
import os
import torch
import json

import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from sacrebleu.metrics import BLEU as SacreBLEU
from rouge_score import rouge_scorer

In [220]:
# SETUP
application = "message_queue_app"
base_model_dir = f"threat_templates/{application}"
models = ["gpt_5_1", "gemini_2_5_pro", "qwen_3_235b_thinking"]
results_dir = f"results/{application}"
stats_dir = f"stats/{application}"

os.makedirs(results_dir, exist_ok=True)
os.makedirs(stats_dir, exist_ok=True)

In [221]:
def load_json(file_path):
    return pd.read_json(file_path)

In [222]:
# Domain models
DOMAIN_MODELS = [
    "cisco-ai/SecureBERT2.0-base",
    "ibm-research/CTI-BERT",
    "basel/ATTACK-BERT"
]

with open(f"{base_model_dir}/assets.json", "r") as f:
    assets = json.load(f)
assets = [asset.lower() for asset in assets]

# Cache the models
domain_embeddings = {name: SentenceTransformer(name) for name in DOMAIN_MODELS}

# Load NLI model
nli_model = pipeline("text-classification", model="roberta-large-mnli", return_all_scores=True)

def compute_text_metrics(base_threat, model_threat):
    # Overlapping assets
    in_base = [asset for asset in assets if asset in base_threat.lower()]
    in_model = [asset for asset in assets if asset in model_threat.lower()]
    if not in_base and not in_model:
        assets_overlap = -1
    elif not in_base:
        assets_overlap = -1
    else: 
        assets_overlap = len(set(in_base).intersection(set(in_model)))

    # Semantic similarity with domain models
    sim_scores = {}
    sim_avg = 0
    for name, model in domain_embeddings.items():
        emb1 = model.encode(base_threat, convert_to_tensor=True)
        emb2 = model.encode(model_threat, convert_to_tensor=True)
        sim_score = util.cos_sim(emb1, emb2).item()
        sim_scores[f"Similarity_{name.split('/')[-1]}"] = float(sim_score)
        sim_avg += sim_score
        
    sim_avg /= len(domain_embeddings)

    # Entailment scores
    nli_input = f"{base_threat} {nli_model.tokenizer.sep_token} {model_threat}"
    nli_result = nli_model(nli_input)[0]  # list of dicts with label & score

    # Extract labels and raw scores
    labels = [r['label'].upper() for r in nli_result]
    raw_scores = torch.tensor([r['score'] for r in nli_result])

    # Softmax to get probabilities
    probs = F.softmax(raw_scores, dim=0)

    # Contrastive entailment: P(ENTAILMENT) - P(CONTRADICTION)
    contrastive_entailment = float(probs[labels.index('ENTAILMENT')] - probs[labels.index('CONTRADICTION')])

    results = {
        "overlap": assets_overlap,
        "contrastive": contrastive_entailment,
        "similarity": sim_avg,
        **sim_scores
    }

    return results

def compute_risk_metric(base_risk, model_risk):
    model_risk = float(model_risk.replace(" out of 10", ""))
    model_risk = model_risk * 10
    if base_risk == "Critical":
        base_risk_no = 100
    elif base_risk == "High":
        base_risk_no = 39
    elif base_risk == "Medium":
        base_risk_no = 24
    else:
        base_risk_no = 10
        
    if model_risk >= 40:
        model_risk_no = 100
    elif model_risk >= 25:
        model_risk_no = 39
    elif model_risk >= 11:
        model_risk_no = 24
    else:
        model_risk_no = 10
        
    if base_risk_no == model_risk_no:
        return 0
    elif model_risk_no > base_risk_no:
        return 1
    else:
        return -1

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 2731299b-5ebb-47c9-aec1-cd387c1ccb1a)')' thrown while requesting HEAD https://huggingface.co/cisco-ai/SecureBERT2.0-base/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name cisco-ai/SecureBERT2.0-base. Creating a new one with mean pooling.
No sentence-transformers model found with name ibm-research/CTI-BERT. Creating a new one with mean pooling.
Some weights of BertModel were not initialized from the model checkpoint at ibm-research/CTI-BERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS

In [223]:
def merge_and_score_threats(base_df, model_df):
    results = []

    for _, row1 in base_df.iterrows():
        for _, row2 in model_df.iterrows():
            # Match Category/Threat Type
            if row1['Category'] == row2['Category']:
                base_threat = row1['Threat']
                model_threat = row2['Threat']

                metrics = compute_text_metrics(base_threat, model_threat)

                results.append({
                    "Category": row1['Category'],
                    "Base": row1['ID'],
                    "Model": row2['ID'],
                    "Base Threat": base_threat,
                    "Model Threat": model_threat,
                    **metrics
                })

    return pd.DataFrame(results)

def merge_and_score_mitigations(base_df, model_df, similar_df):
    results = []

    for _, row1 in similar_df.iterrows():
        base_row = base_df[base_df['ID'] == row1['Base']].iloc[0]
        model_row = model_df[model_df['ID'] == row1['Model']].iloc[0]

        base_mitigation = base_row['Mitigation']
        model_mitigation = model_row['Mitigation']

        metrics = compute_text_metrics(base_mitigation, model_mitigation)

        results.append({
            "Category": base_row['Category'],
            "Base": base_row['ID'],
            "Model": model_row['ID'],
            "Base Mitigation": base_mitigation,
            "Model Mitigation": model_mitigation,
            **metrics
        })

    return pd.DataFrame(results)

def merge_and_score_risk(base_df, model_df, similar_df):
    results = []

    for _, row1 in similar_df.iterrows():
        base_row = base_df[base_df['ID'] == row1['Base']].iloc[0]
        model_row = model_df[model_df['ID'] == row1['Model']].iloc[0]

        base_risk = base_row['Risk']
        model_risk = model_row['Risk']

        metric = compute_risk_metric(base_risk, model_risk)

        results.append({
            "Category": base_row['Category'],
            "Base": base_row['ID'],
            "Model": model_row['ID'],
            "Base Risk": base_risk,
            "Model Risk": model_risk,
            "Risk": metric
        })

    return pd.DataFrame(results)

In [224]:
def visualize_scores(df, metrics, title="Threat Similarity Metrics"):

    # Extract values for the metrics
    data = df[metrics].values

    num_items = data.shape[0]
    num_metrics = len(metrics)

    # Set bar positions
    x = np.arange(num_metrics)
    width = 0.8 / num_items  # adjust bar width based on number of rows

    plt.figure(figsize=(12, 6))

    for i in range(num_items):
        plt.bar(
            x + i * width,
            data[i],
            width,
            label=f"Pair {i+1}"
        )

    # Formatting
    plt.xticks(x + width * (num_items - 1) / 2, metrics, rotation=45)
    plt.ylabel("Score")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [225]:
def get_similar_pairs_all_metrics(results_df):
    all_similar = results_df[
        ((results_df['overlap'] >= 1) | (results_df['overlap'] == -1)) &
        (results_df['similarity'] >= 0.5) &
        (results_df['contrastive'] >= 0)
    ]
    
    similar = all_similar.copy()
    similar["score"] = (similar[["overlap", "contrastive", "similarity"]].sum(axis=1))
    similar = similar.sort_values(by='score', ascending=False).drop_duplicates(subset=['Base'], keep='first')
    
    return all_similar, similar

def get_similar_pairs_metrics(results_df):
    similar = results_df[
        ((results_df['overlap'] >= 1) | (results_df['overlap'] == -1)) &
        (results_df['similarity'] >= 0.5) &
        (results_df['contrastive'] >= 0)
    ]
    
    return similar

In [226]:
for model in models:
    original_df = load_json(f"{base_model_dir}/threat_model.json")
    model_df = load_json(f"{results_dir}/{model}_stridegpt_model.json")
    merged_model_df = merge_and_score_threats(original_df, model_df)
    merged_model_df.to_json(f"{stats_dir}/{model}_scored_threats.json", orient="records", indent=2)
    all_similar_threats, similar_threats = get_similar_pairs_all_metrics(merged_model_df)
    similar_threats.to_json(f"{stats_dir}/{model}_similar_threats.json", orient="records", indent=2)
    all_similar_threats.to_json(f"{stats_dir}/{model}_all_similar_threats.json", orient="records", indent=2)

In [227]:
similar_threats_dfs = {}
similar_mitigations_dfs = {}
similar_risk_dfs = {}
common_threats = set()
same = {}
for model in models:
    same[model] = {}
    # Load stats
    similar_threats = pd.read_json(f"{stats_dir}/{model}_similar_threats.json")
    all_similar_threats = pd.read_json(f"{stats_dir}/{model}_all_similar_threats.json")

    same[model]["threats"] = len(similar_threats)
    same[model]["total_similar_pairs"] = len(all_similar_threats)
    if len(common_threats) == 0:
        common_threats = set(similar_threats['Base'])
    else:
        common_threats.intersection(set(similar_threats['Base']))

    if len(similar_threats) > 0:
        original_df = load_json(f"{base_model_dir}/threat_model.json")
        model_df = load_json(f"{results_dir}/{model}_stridegpt_model.json")
        mitigation_df = merge_and_score_mitigations(original_df, model_df, similar_threats)
        mitigation_df.to_json(f"{stats_dir}/{model}_scored_mitigations.json", orient="records", indent=2)

        risk_df = merge_and_score_risk(original_df, model_df, similar_threats)
        risk_df.to_json(f"{stats_dir}/{model}_scored_risks.json", orient="records", indent=2)
        
        similar_mitigation = get_similar_pairs_metrics(mitigation_df)
        similar_mitigations_dfs[model] = similar_mitigation
        similar_mitigation.to_json(f"{stats_dir}/{model}_similar_mitigations.json", orient="records", indent=2)
        same[model]["mitigations"] = len(similar_mitigation)

        same[model]["risk_same"] = len(risk_df[risk_df['Risk'] == 0])
        same[model]["risk_more"] = len(risk_df[risk_df['Risk'] == 1])
        same[model]["risk_less"] = len(risk_df[risk_df['Risk'] == -1])
    else:
        similar_mitigation = pd.DataFrame()
        risk_df = pd.DataFrame()
        
    print(f"Model: {model}")
    print(f"Threats in base: {len(original_df)}")
    print(f"Threats in model: {len(model_df)}")
    print(f"Unique similar threats in model: {len(similar_threats)}")
    print(f"Unique duplicates threats in model: {len(all_similar_threats)-len(similar_threats)}")
    print(f"Similar mitigations found: {len(similar_mitigation)}")
    print(f"Same risks found: {len(risk_df[risk_df['Risk'] == 0])}")
    print(f"Higher risks found: {len(risk_df[risk_df['Risk'] == 1])}")
    print(f"Lower risks found: {len(risk_df[risk_df['Risk'] == -1])}")
    print("-" * 50)
    
print(f"Threats in base model: {len(original_df)}")
print(f"Total number of overlapping threats between the different models': {len(common_threats)}")

with open (f"{stats_dir}/similarity_summary.json", "w") as f:
    json.dump(same, f, indent=2)

Model: gpt_5_1
Threats in base: 13
Threats in model: 22
Unique similar threats in model: 13
Unique duplicates threats in model: 18
Similar mitigations found: 7
Same risks found: 0
Higher risks found: 13
Lower risks found: 0
--------------------------------------------------
Model: gemini_2_5_pro
Threats in base: 13
Threats in model: 16
Unique similar threats in model: 9
Unique duplicates threats in model: 5
Similar mitigations found: 2
Same risks found: 0
Higher risks found: 9
Lower risks found: 0
--------------------------------------------------
Model: qwen_3_235b_thinking
Threats in base: 13
Threats in model: 22
Unique similar threats in model: 13
Unique duplicates threats in model: 23
Similar mitigations found: 8
Same risks found: 0
Higher risks found: 13
Lower risks found: 0
--------------------------------------------------
Threats in base model: 13
Total number of overlapping threats between the different models': 13


In [228]:
import re
import pandas as pd

with open(f"{base_model_dir}/assets.json", "r") as f:
    assets = json.load(f)
assets = set(assets)
print(assets)

for model in models:
    model_df = load_json(f"{results_dir}/{model}_stridegpt_model.json")
    texts = model_df['Threat'].astype(str).tolist()

    pattern = r'\b(?:[A-Z][a-z]+(?:\s+|$)){2,}'

    matches = []
    for text in texts:
        found = re.findall(pattern, text)
        matches.extend([m.strip() for m in found])

    unique_names = set(matches)
    print("-" * 50)
    print(f"Model: {model}")
    print(unique_names - assets)

{'Message Queue', 'Web Application', 'Background Worker Config', 'Web Application Config', 'Browser', 'Database', 'Background Worker'}
--------------------------------------------------
Model: gpt_5_1
{'The Background Worker', 'Basic Auth', 'The Web Application', 'If Basic Authentication', 'Basic Authentication'}
--------------------------------------------------
Model: gemini_2_5_pro
{'The Background Worker', 'The Web Application'}
--------------------------------------------------
Model: qwen_3_235b_thinking
{'The Database', 'The Background Worker', 'The Web Application'}
